<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎬 LTX-2.3 22B Distilled 1.1 Q4 Video Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition - Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0;'>Text-to-Video • Image-to-Video • First & Last Frame, with audio | Wan2GP Engine + mmgp</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

### What is this notebook?

**Text-to-Video & Image-to-Video Generation** with synchronized audio, using **LTX-2.3 22B Distilled 1.1 (GGUF Q4_K_M)**.

| Spec | Configuration |
|---|---|
| **GPU** | Kaggle GPU T4 x2 (also runs on a single T4, slower) |
| **Model** | LTX-2.3 22B Distilled 1.1 Q4_K_M + Gemma 3 12B text encoder |
| **Pipeline** | Two-stage: 8 steps half-res → 2x spatial upscale → 3 steps refine → FP32 VAE |
| **GPU 0** | Transformer + VAE decoder |
| **GPU 1** | Gemma 3 text encoder + spatial upsampler + video encoder |
| **Modes** | Text-to-Video, Image-to-Video, First Frame, Last Frame, First + Last Frame |
| **Output** | Saved to `/kaggle/working/outputs` (Kaggle Output panel) |

### Quick Start
1. **Settings → Accelerator → GPU T4 x2**
2. **Turn on Internet** in Settings sidebar
3. Run all cells in order
4. Open the **Gradio** link, or the **Cloudflare** tunnel link if Gradio's link does not load
5. Use detailed prompts (subject, action, setting, camera, lighting, sound) for the best results

---
## Step 1: Environment Setup

Optimizes memory for Kaggle T4 GPU (~30GB RAM, 16GB VRAM).

In [1]:
import os, gc, psutil

print('=== Kaggle T4 Environment Setup ===')
print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total, {psutil.virtual_memory().available / 1024**3:.1f} GB available')

os.system('echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1')
os.system('echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1')
gc.collect()

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['MALLOC_TRIM_THRESHOLD_'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Environment optimized!')
print('   Kaggle has ~30GB RAM, no swap needed.')

=== Kaggle T4 Environment Setup ===
RAM: 31.3 GB total, 30.1 GB available
✅ Environment optimized!
   Kaggle has ~30GB RAM - no swap needed.


---
## Step 2: Clone Wan2GP, Install Dependencies & Apply Patches

In [2]:
import os
import sys
import subprocess

try:
    smi = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True).strip()
    print(f'✅ GPU Detected:\n{smi}')
except Exception:
    print('WARNING: No GPU. Go to Settings → Accelerator → GPU T4 x2')

REPO_DIR = 'Wan2GP'
if not os.path.exists(REPO_DIR):
    print('\n📥 Cloning Wan2GP repository...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/DeepBeepMeep/Wan2GP.git', REPO_DIR], check=True)
else:
    # Drop runtime patches from earlier runs; the current ones are re-applied below
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '-q', '--hard'], check=True)
    print('\n📂 Wan2GP repository already present.')

# Wan2GP's requirements pin an onnxruntime-gpu dev build that is not on PyPI (it makes the whole
# install fail part-way), so that one line uses the released package.
print('📦 Installing Wan2GP requirements...')
with open(f'{REPO_DIR}/requirements.txt') as f:
    reqs = [('onnxruntime-gpu>=1.22.0' if line.startswith('onnxruntime-gpu==') and 'dev' in line else line.rstrip('\n'))
            for line in f]
with open('wan2gp_requirements.txt', 'w') as f:
    f.write('\n'.join(reqs) + '\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--timeout', '120', '--retries', '5',
                '-q', '--no-warn-conflicts', '-r', 'wan2gp_requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--timeout', '120', '--retries', '5',
                '-q', '--no-warn-conflicts', 'gradio>=5.0.0', 'gguf', 'soundfile', 'einops',
                'sentencepiece', 'hf_transfer', 'accelerate'], check=True)
print('✅ Dependencies installed.')

# 5. Apply Wan2GP Core Runtime Patches
print('🔧 Applying Wan2GP runtime patches...')

# Patch 1: ltx2.py runtime fixes
ltx2_py = os.path.join(REPO_DIR, 'models/ltx2/ltx2.py')
if os.path.exists(ltx2_py):
    with open(ltx2_py, 'r') as f:
        content = f.read()
    old_line1 = 'input_video_strength = max(0.0, min(1.0, input_video_strength))'
    new_line1 = 'input_video_strength = max(0.0, min(1.0, input_video_strength)) if input_video_strength is not None else 1.0'
    if old_line1 in content:
        content = content.replace(old_line1, new_line1)
    old_line2 = 'source = source.to(device=self.device, dtype=torch.bfloat16)'
    new_line2 = 'source = source.to(device=self.device, dtype=self.dtype)'
    if old_line2 in content:
        content = content.replace(old_line2, new_line2)
    with open(ltx2_py, 'w') as f:
        f.write(content)
    print('  ✓ Patched Wan2GP ltx2.py routines')

# Patch 2: Spatial Upsampler Dtype Auto-Alignment
upsampler_file = os.path.join(REPO_DIR, 'models/ltx2/ltx_core/model/upsampler/model.py')
if os.path.exists(upsampler_file):
    with open(upsampler_file, 'r') as f:
        ucontent = f.read()
    old_up = 'latent = upsampler(latent)'
    new_up = ('upsampler_dtype = next(upsampler.parameters()).dtype if any(True for _ in upsampler.parameters()) else torch.float16\n'
              '    if latent.dtype != upsampler_dtype:\n'
              '        latent = latent.to(upsampler_dtype)\n'
              '    latent = upsampler(latent)')
    if old_up in ucontent:
        ucontent = ucontent.replace(old_up, new_up)
    with open(upsampler_file, 'w') as f:
        f.write(ucontent)
    print('  ✓ Patched Wan2GP spatial upsampler auto-alignment')

# Patch 3: Video VAE Dtype Auto-Alignment
vae_file = os.path.join(REPO_DIR, 'models/ltx2/ltx_core/model/video_vae/video_vae.py')
if os.path.exists(vae_file):
    with open(vae_file, 'r') as f:
        vcontent = f.read()
    old_v = 'sample = self.conv_in(sample, causal=self.causal)'
    new_v = ('conv_param = next(self.conv_in.parameters(), None)\n'
             '        if conv_param is not None and sample.dtype != conv_param.dtype:\n'
             '            sample = sample.to(conv_param.dtype)\n'
             '        sample = self.conv_in(sample, causal=self.causal)')
    if old_v in vcontent:
        vcontent = vcontent.replace(old_v, new_v)
    old_enc = 'sample = self.conv_in(sample)'
    new_enc = ('enc_param = next(self.conv_in.parameters(), None)\n'
               '        if enc_param is not None and sample.dtype != enc_param.dtype:\n'
               '            sample = sample.to(enc_param.dtype)\n'
               '        sample = self.conv_in(sample)')
    if old_enc in vcontent:
        vcontent = vcontent.replace(old_enc, new_enc)
    with open(vae_file, 'w') as f:
        f.write(vcontent)
    print('  ✓ Patched Wan2GP video VAE auto-alignment')

# Patch 4: Audio VAE Dtype Auto-Alignment
audio_vae_file = os.path.join(REPO_DIR, 'models/ltx2/ltx_core/model/audio_vae/audio_vae.py')
if os.path.exists(audio_vae_file):
    with open(audio_vae_file, 'r') as f:
        acontent = f.read()
    old_a = 'decoded_audio = audio_decoder(latent)'
    new_a = ('decoder_param = next(audio_decoder.parameters(), None)\n'
             '    if decoder_param is not None and latent.dtype != decoder_param.dtype:\n'
             '        latent = latent.to(decoder_param.dtype)\n'
             '    decoded_audio = audio_decoder(latent)\n'
             '    vocoder_param = next(vocoder.parameters(), None)\n'
             '    if vocoder_param is not None and decoded_audio.dtype != vocoder_param.dtype:\n'
             '        decoded_audio = decoded_audio.to(vocoder_param.dtype)')
    if old_a in acontent:
        acontent = acontent.replace(old_a, new_a)
    with open(audio_vae_file, 'w') as f:
        f.write(acontent)
    print('  ✓ Patched Wan2GP audio VAE auto-alignment')

# Patch 5: Vocoder STFT & Filter Dtype Auto-Alignment
vocoder_file = os.path.join(REPO_DIR, 'models/ltx2/ltx_core/model/audio_vae/vocoder.py')
if os.path.exists(vocoder_file):
    with open(vocoder_file, 'r') as f:
        voc_content = f.read()
    old_stft = 'spec = F.conv1d(y, self.forward_basis, stride=self.hop_length, padding=0)'
    new_stft = ('forward_basis = self.forward_basis.to(dtype=y.dtype, device=y.device) if self.forward_basis.dtype != y.dtype else self.forward_basis\n'
                '        spec = F.conv1d(y, forward_basis, stride=self.hop_length, padding=0)')
    if old_stft in voc_content:
        voc_content = voc_content.replace(old_stft, new_stft)
    old_low = 'return F.conv1d(x, self.filter.expand(n_channels, -1, -1), stride=self.stride, groups=n_channels)'
    new_low = ('filt = self.filter.to(dtype=x.dtype, device=x.device) if self.filter.dtype != x.dtype else self.filter\n'
               '        return F.conv1d(x, filt.expand(n_channels, -1, -1), stride=self.stride, groups=n_channels)')
    if old_low in voc_content:
        voc_content = voc_content.replace(old_low, new_low)
    with open(vocoder_file, 'w') as f:
        f.write(voc_content)
    print('  ✓ Patched Wan2GP vocoder auto-alignment')

# Patch 6: Wan2GP Base Text Encoder Device Alignment (Fixes Multi-GPU device mismatch)
# Gemma's hidden states stay in BF16: the feature extractor squares them for RMS normalization, and in
# FP16 any value above 256 overflows (inf -> NaN text features -> black video).
base_enc_file = os.path.join(REPO_DIR, 'models/ltx2/ltx_core/text_encoders/gemma/encoders/base_encoder.py')
if os.path.exists(base_enc_file):
    with open(base_enc_file, 'r') as f:
        bcontent = f.read()
    old_b = 'attention_mask = item.attention_mask.to("cuda")\n        encoded_video_input, encoded_audio_input = _apply_feature_extractor(\n            item.hidden_states,'
    new_b = ('target_dev = next(feature_extractor_linear.parameters()).device if any(True for _ in feature_extractor_linear.parameters()) else torch.device("cuda:0")\n'
             '        attention_mask = item.attention_mask.to(target_dev)\n'
             '        hidden_states = tuple(h.to(device=target_dev) for h in item.hidden_states)\n'
             '        encoded_video_input, encoded_audio_input = _apply_feature_extractor(\n'
             '            hidden_states,')
    if old_b in bcontent:
        bcontent = bcontent.replace(old_b, new_b)
        with open(base_enc_file, 'w') as f:
            f.write(bcontent)
        print('  ✓ Patched Wan2GP base_encoder device & dtype auto-alignment')

# Patch 7: Wan2GP Gemma Feature Extractor & Connector Dtype Auto-alignment
fe_file = os.path.join(REPO_DIR, 'models/ltx2/ltx_core/text_encoders/gemma/feature_extractor.py')
if os.path.exists(fe_file):
    with open(fe_file, 'r') as f:
        fe_content = f.read()
    old_fe = 'normed = _norm_and_concat_per_token_rms(encoded, attention_mask).to(encoded.dtype)\n            v_dim = self.video_aggregate_embed.out_features'
    new_fe = ('v_target_dtype = next(self.video_aggregate_embed.parameters()).dtype if any(True for _ in self.video_aggregate_embed.parameters()) else encoded.dtype\n'
              '            v_target_dev = next(self.video_aggregate_embed.parameters()).device if any(True for _ in self.video_aggregate_embed.parameters()) else encoded.device\n'
              '            normed = _norm_and_concat_per_token_rms(encoded.to(device=v_target_dev), attention_mask.to(v_target_dev)).to(dtype=v_target_dtype)\n'
              '            v_dim = self.video_aggregate_embed.out_features')
    if old_fe in fe_content:
        fe_content = fe_content.replace(old_fe, new_fe)
    old_fe_aud = 'audio = self.audio_aggregate_embed(_rescale_norm(normed, a_dim, self.embedding_dim))'
    new_fe_aud = ('a_target_dtype = next(self.audio_aggregate_embed.parameters()).dtype if any(True for _ in self.audio_aggregate_embed.parameters()) else v_target_dtype\n'
                  '                normed_a = normed.to(dtype=a_target_dtype) if a_target_dtype != v_target_dtype else normed\n'
                  '                audio = self.audio_aggregate_embed(_rescale_norm(normed_a, a_dim, self.embedding_dim))')
    if old_fe_aud in fe_content:
        fe_content = fe_content.replace(old_fe_aud, new_fe_aud)
    with open(fe_file, 'w') as f:
        f.write(fe_content)
    print('  ✓ Patched Wan2GP feature_extractor dtype auto-alignment')

conn_file = os.path.join(REPO_DIR, 'models/ltx2/ltx_core/text_encoders/gemma/embeddings_connector.py')
if os.path.exists(conn_file):
    with open(conn_file, 'r') as f:
        conn_content = f.read()
    old_reg = 'learnable_registers = torch.tile(self.learnable_registers, (num_registers_duplications, 1))'
    new_reg = 'learnable_registers = torch.tile(self.learnable_registers.to(device=hidden_states.device, dtype=hidden_states.dtype), (num_registers_duplications, 1))'
    if old_reg in conn_content:
        conn_content = conn_content.replace(old_reg, new_reg)
        with open(conn_file, 'w') as f:
            f.write(conn_content)
        print('  ✓ Patched Wan2GP embeddings_connector register auto-alignment')

print('✅ Wan2GP setup complete!')

Fri Sep 25 01:16:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## Step 3: Download All Required Models

Downloads GGUF Q4_K_M transformer + companion files. Large files → `/kaggle/tmp` (symlinked).

In [3]:
import os
from huggingface_hub import hf_hub_download

REPO = 'Abiray/LTX-2.3-22B-DISTILLED-1.1-GGUF'
COMPANION_REPO = 'DeepBeepMeep/LTX-2'
MODEL_DIR = 'Wan2GP/models'
TMP_DIR = '/kaggle/tmp/models'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

# Distilled 1.1 Q4_K_M transformer from Abiray
TRANSFORMER_FILE    = 'LTX-2.3-22B-distilled-1.1-Q4_K_M.gguf'
TRANSFORMER_RENAMED = 'ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf'

# Download transformer from Abiray
dest = os.path.join(MODEL_DIR, TRANSFORMER_RENAMED)
if os.path.exists(dest):
    print(f'  \u2713 Already exists: {TRANSFORMER_RENAMED}')
else:
    print(f'Downloading {TRANSFORMER_FILE} from {REPO} \u2192 /kaggle/tmp ...')
    hf_hub_download(repo_id=REPO, filename=TRANSFORMER_FILE, local_dir=TMP_DIR)
    actual = os.path.join(TMP_DIR, TRANSFORMER_FILE)
    os.symlink(actual, dest)
    print(f'  \u2713 {TRANSFORMER_RENAMED} (symlinked)')

# Companion large files from DeepBeepMeep
LARGE_FILES_COMPANION = [
    'ltx-2.3-22b_embeddings_connector.safetensors',
    'ltx-2.3-22b_text_embedding_projection.safetensors',
    'ltx-2.3-22b_vae.safetensors',
]

for f in LARGE_FILES_COMPANION:
    dest = os.path.join(MODEL_DIR, f)
    if os.path.exists(dest):
        print(f'  ✓ Already exists: {f}')
        continue
    print(f'Downloading {f} → /kaggle/tmp ...')
    hf_hub_download(repo_id=COMPANION_REPO, filename=f, local_dir=TMP_DIR)
    actual = os.path.join(TMP_DIR, f)
    os.symlink(actual, dest)
    print(f'  ✓ {f} (symlinked)')

# Small files download directly
SMALL_FILES = [
    'ltx-2.3-22b_audio_vae.safetensors',                 # 107 MB
    'ltx-2.3-22b_vocoder.safetensors',                   # 258 MB
    'ltx-2.3-spatial-upscaler-x2-1.1.safetensors',       # 996 MB
]

for f in SMALL_FILES:
    dest = os.path.join(MODEL_DIR, f)
    if os.path.exists(dest):
        print(f'  ✓ Already exists: {f}')
        continue
    print(f'Downloading {f}...')
    hf_hub_download(repo_id=COMPANION_REPO, filename=f, local_dir=MODEL_DIR)
    print(f'  ✓ {f}')

# Gemma text encoder → /kaggle/tmp
GEMMA_FOLDER = 'gemma-3-12b-it-qat-q4_0-unquantized'
GEMMA_FILES = [
    'gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors',
    'added_tokens.json', 'chat_template.json', 'config_light.json',
    'generation_config.json', 'preprocessor_config.json', 'processor_config.json',
    'special_tokens_map.json', 'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json',
]

gemma_dest = os.path.join(MODEL_DIR, GEMMA_FOLDER)
gemma_tmp = os.path.join(TMP_DIR, GEMMA_FOLDER)
if os.path.exists(gemma_dest):
    print(f'  ✓ Already exists: {GEMMA_FOLDER}/')
else:
    os.makedirs(gemma_tmp, exist_ok=True)
    for gf in GEMMA_FILES:
        tmp_file = os.path.join(gemma_tmp, gf)
        if os.path.exists(tmp_file): continue
        print(f'Downloading gemma/{gf} → /kaggle/tmp ...')
        hf_hub_download(repo_id=COMPANION_REPO, filename=f'{GEMMA_FOLDER}/{gf}', local_dir=TMP_DIR)
    os.symlink(gemma_tmp, gemma_dest)
    print(f'  ✓ {GEMMA_FOLDER}/ (symlinked)')

# Cleanup HF cache
import shutil
for d in [os.path.join(MODEL_DIR, '.cache'), os.path.join(TMP_DIR, '.cache')]:
    if os.path.exists(d): shutil.rmtree(d)

os.system('df -h /kaggle/working /kaggle/tmp')
print('\n✅ All downloads complete!')

LTX-2.3-22B-distilled-1.1-Q4_K_M.gguf:   0%|          | 0.00/17.8G [00:00<?, ?B/s]

  ✓ ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf (symlinked)


ltx-2.3-22b-distilled-lora-384.safetenso(…):   0%|          | 0.00/7.61G [00:00<?, ?B/s]

  ✓ ltx-2.3-22b-distilled-lora-384.safetensors (symlinked)


ltx-2.3-22b_embeddings_connector.safeten(…):   0%|          | 0.00/4.03G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

gemma-3-12b-it-qat-q4_0-unquantized/toke(…):   0%|          | 0.00/33.4M [00:00<?, ?B/s]

gemma-3-12b-it-qat-q4_0-unquantized/toke(…):   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

  ✓ gemma-3-12b-it-qat-q4_0-unquantized/ (symlinked)
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G  1.7G   18G   9% /kaggle/working
overlay         8.0T  7.0T  1.1T  87% /

✅ All downloads complete!


---
## Step 4: Write the Text/Image-to-Video Script

Creates `run_ltx_t2v.py`: model loading, dual-T4 layout, FP32 VAE, generation and the branded Gradio UI.

In [4]:
%%writefile run_ltx_t2v.py
import gc
import os
import sys
import json
import random
import tempfile
import glob
import time
import traceback
import numpy as np
import subprocess
import psutil
import soundfile as sf
from PIL import Image

# 1. Environment & Performance Settings
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.6"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"

# ---- bootstrap Wan2GP ----
WAN2GP_DIR = os.path.abspath("Wan2GP")
# Videos are saved in the working folder (/kaggle/working/outputs), visible in Kaggle's Output panel
OUTPUT_DIR = os.path.join(os.path.dirname(WAN2GP_DIR), "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ.setdefault("GRADIO_TEMP_DIR", os.path.join(OUTPUT_DIR, ".gradio_cache"))
if WAN2GP_DIR not in sys.path:
    sys.path.insert(0, WAN2GP_DIR)
if os.path.exists(WAN2GP_DIR):
    os.chdir(WAN2GP_DIR)

import torch
import gradio as gr
from shared.utils.audio_video import save_video

# Detect Hardware
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
_GPU_SM = torch.cuda.get_device_capability(0) if NUM_GPUS > 0 else (0, 0)
_IS_SM60 = (_GPU_SM[0] == 6)

print("=" * 65)
print("🎬 LTX-2.3 22B Distilled 1.1 - Text & Image-to-Video Engine")
print("📺 Created by: AIQUEST Academy")
print(f"🎮 Accelerator: {'Dual GPU (T4 x2)' if NUM_GPUS > 1 else 'Single GPU'} | Compute: sm_{_GPU_SM[0]}{_GPU_SM[1]}")
print("=" * 65)

# CUDA Attention & Convolution Optimizations for Tesla T4 (Turing sm_75)
torch.backends.cuda.enable_flash_sdp(False)          # sm_75 does not support FlashAttention v2
torch.backends.cuda.enable_mem_efficient_sdp(True)   # Fast memory-efficient attention (Cutlass kernel)
torch.backends.cuda.enable_math_sdp(True)            # Safe fallback
torch.backends.cudnn.benchmark = False               # Autotuning FP32 conv3d on a T4 stalls the VAE for 10+ minutes

# ==== 1. GEMMA 3 SDPA & ROPE DTYPE PATCH ====
def _patch_gemma3_attention():
    """
    Fixes RuntimeError: Expected query, key, and value to have the same dtype,
    got query.dtype: float key.dtype: float and value.dtype: c10::BFloat16.
    """
    try:
        import transformers.models.gemma3.modeling_gemma3 as g3_mod
        _orig_rope = g3_mod.apply_rotary_pos_emb

        def _safe_apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
            if cos.dtype != q.dtype:
                cos = cos.to(dtype=q.dtype)
            if sin.dtype != q.dtype:
                sin = sin.to(dtype=q.dtype)
            return _orig_rope(q, k, cos, sin, unsqueeze_dim=unsqueeze_dim)

        g3_mod.apply_rotary_pos_emb = _safe_apply_rotary_pos_emb
        print("  [Gemma3 Patch] ✅ apply_rotary_pos_emb patched (RoPE dtype aligned with query)")
    except Exception as e:
        print(f"  [Gemma3 Patch] ⚠️ Could not patch apply_rotary_pos_emb: {e}")

    try:
        import transformers.integrations.sdpa_attention as sdpa_mod
        _orig_sdpa = sdpa_mod.sdpa_attention_forward

        def _safe_sdpa_attention_forward(module, query, key, value, *args, **kwargs):
            if key.dtype != query.dtype:
                key = key.to(dtype=query.dtype)
            if value.dtype != query.dtype:
                value = value.to(dtype=query.dtype)
            return _orig_sdpa(module, query, key, value, *args, **kwargs)

        sdpa_mod.sdpa_attention_forward = _safe_sdpa_attention_forward
        print("  [SDPA Patch] ✅ sdpa_attention_forward patched (query/key/value dtype auto-aligned)")
    except Exception as e:
        print(f"  [SDPA Patch] ⚠️ Could not patch sdpa_attention_forward: {e}")

_patch_gemma3_attention()

# ==== 2. GGUF EXTENSION HANDLER & CONFIG PATCH ====
def _register_gguf_handler():
    """Register the GGUF handler with mmgp's quant_router."""
    try:
        import shared.qtypes.gguf
        print("  [GGUF] ✅ Extension handler registered with mmgp (Wan2GP native)")
    except Exception as e:
        print(f"  [GGUF] ⚠️ Could not register gguf handler: {e}")

def _patch_ltx2_config_loading():
    """Patch _load_config_from_checkpoint to handle GGUF metadata gracefully."""
    import models.ltx2.ltx2 as ltx2_mod
    _original = ltx2_mod._load_config_from_checkpoint

    def _patched(path, fallback_config_path=None):
        from mmgp import quant_router
        if isinstance(path, (list, tuple)):
            path = path[0] if path else ""
        if not path:
            return {}
        try:
            _, metadata = quant_router.load_metadata_state_dict(path)
            if metadata:
                config_raw = metadata.get("config")
                if config_raw:
                    config = ltx2_mod._normalize_config(config_raw)
                    if config:
                        return config
        except Exception:
            pass
        if fallback_config_path and os.path.isfile(fallback_config_path):
            try:
                with open(fallback_config_path, "r", encoding="utf-8") as f:
                    config = ltx2_mod._normalize_config(json.load(f))
                    if config:
                        print(f"  [GGUF Patch] ✅ Config loaded from {os.path.basename(fallback_config_path)}")
                        return config
            except Exception:
                pass
        return {}

    ltx2_mod._load_config_from_checkpoint = _patched
    print("  [GGUF Patch] ✅ Config loading patched for GGUF")

_register_gguf_handler()
_patch_ltx2_config_loading()

# ==== 3. GPU SPECIFICATIONS & MEMORY STATS ====
if torch.cuda.is_available():
    print(f"Primary GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    if NUM_GPUS > 1:
        print(f"Secondary GPU: {torch.cuda.get_device_name(1)} (VRAM: {torch.cuda.get_device_properties(1).total_memory / 1024**3:.1f} GB)")
ram = psutil.virtual_memory()
print(f"System RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")
sys.stdout.flush()

# ==== 4. LOAD LTX-2.3 MODEL COMPONENTS ====
print("\nLoading LTX-2.3 22B Distilled 1.1 (GGUF Q4_K_M)...")
sys.stdout.flush()

from mmgp import offload
from shared.utils import files_locator as fl
from models.ltx2.ltx2_handler import family_handler

fl.set_checkpoints_paths(["models", "ckpts", "."])

base_model_type = "ltx2_22B"
model_def = {"ltx2_pipeline": "distilled"}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

gemma_folder = "models/gemma-3-12b-it-qat-q4_0-unquantized"
gemma_files = sorted(glob.glob(os.path.join(gemma_folder, "*.safetensors")))
quanto_files = [f for f in gemma_files if "quanto" in f]
text_encoder_file = quanto_files[0] if quanto_files else (gemma_files[0] if gemma_files else None)
if not text_encoder_file:
    raise FileNotFoundError(f"No .safetensors in {gemma_folder}. Check download cell.")
print(f"  Text encoder: {os.path.basename(text_encoder_file)}")

transformer_path = os.path.join("models", "ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf")
if not os.path.isfile(transformer_path):
    raise FileNotFoundError(f"{transformer_path} missing. Check download cell.")
print(f"  Transformer : {os.path.basename(transformer_path)}")
sys.stdout.flush()

# FP16 transformer: the T4's tensor cores make it ~3x faster than BF16 (the original May setup).
# Gemma's hidden states are kept in BF16 (see the dual-GPU section); a NaN guard after Stage 1 stops
# early if anything overflows. The video VAE + upsampler run in FP32 (converted below).
MODEL_DTYPE = torch.float16
VAE_DTYPE   = torch.float32

ltx2_model, pipe = family_handler.load_model(
    model_filename=transformer_path,
    model_type="ltx2_22B_distilled",
    base_model_type=base_model_type,
    model_def=model_def,
    dtype=MODEL_DTYPE,
    VAE_dtype=VAE_DTYPE,
    text_encoder_filename=text_encoder_file,
)

# Align Gemma embed_tokens to bfloat16 to avoid float32 inputs_embeds
try:
    _g_embed = ltx2_model.text_encoder.model.model.embed_tokens
    _g_embed.to(torch.bfloat16)
except Exception:
    try:
        ltx2_model.text_encoder.model.embed_tokens.to(torch.bfloat16)
    except Exception:
        pass

# ==== 4b. FP32 VIDEO VAE, DTYPE-SAFE FORWARDS & STAGE PROBES ====
# The LTX video VAE + latent upsampler must run in FP32 on a T4: in FP16/BF16 the decoded video
# collapses to flat grey with neon edges. The pipeline feeds them BF16 latents (the cause of
# "Input type (BFloat16) and bias type (Half)"), so their forwards cast the input to the weight
# dtype/device and the encoder hands its result back in the caller's dtype and device.
for _key in ("video_decoder", "video_encoder", "spatial_upsampler"):
    _mod = getattr(ltx2_model, _key, None)
    if _mod is None:
        continue
    _mod.to(torch.float32)
    _mod._model_dtype = torch.float32
    for _m in _mod.modules():
        _m._lock_dtype = torch.float32
print("  [VAE] ✅ Video VAE + spatial upsampler converted to FP32")

import models.ltx2.ltx_core.model.video_vae.video_vae as _video_vae_mod
_orig_video_decoder_forward = _video_vae_mod.VideoDecoder.forward
_orig_video_encoder_forward = _video_vae_mod.VideoEncoder.forward

def _fp32_video_decoder_forward(self, sample, *args, **kwargs):
    param = next(self.parameters(), None)
    if param is not None and sample.dtype != param.dtype:
        sample = sample.to(param.dtype)
    return _orig_video_decoder_forward(self, sample, *args, **kwargs)

def _fp32_video_encoder_forward(self, sample, *args, **kwargs):
    in_dtype, in_device = sample.dtype, sample.device
    param = next(self.parameters(), None)
    if param is not None and param.device.type == "cuda" and sample.device != param.device:
        sample = sample.to(param.device)
    if param is not None and sample.dtype != param.dtype:
        sample = sample.to(param.dtype)
    with torch.cuda.device(sample.device if sample.is_cuda else torch.cuda.current_device()):
        out = _orig_video_encoder_forward(self, sample, *args, **kwargs)
    if torch.is_tensor(out) and out.is_floating_point():
        out = out.to(device=in_device, dtype=in_dtype if in_dtype.is_floating_point else out.dtype)
    return out

_video_vae_mod.VideoDecoder.forward = _fp32_video_decoder_forward
_video_vae_mod.VideoEncoder.forward = _fp32_video_encoder_forward

# distilled.py binds these helpers by name at import, so wrap them inside that module
import models.ltx2.ltx_pipelines.distilled as _distilled_mod
from models.ltx2.ltx_core.model.video_vae.tiling import TilingConfig, SpatialTilingConfig, TemporalTilingConfig
_orig_pipeline_upsample = _distilled_mod.upsample_video
_orig_pipeline_decode = _distilled_mod.vae_decode_video_to_tensor
UPS_ON_GPU1 = False  # set below once the upsampler + encoder are moved to GPU 1

def _latent_stats(tag, t):
    try:
        t = t[0] if isinstance(t, (list, tuple)) else t
        f = t.detach().float()
        nonfinite = int((~torch.isfinite(f)).sum().item())
        f = torch.nan_to_num(f)
        print(f"  🔬 {tag}: shape={tuple(t.shape)} mean={f.mean().item():+.3f} std={f.std().item():.3f} nonfinite={nonfinite}")
        sys.stdout.flush()
        return nonfinite
    except Exception as _e:
        print(f"  🔬 {tag}: stats unavailable ({_e})")
        return 0

def _probed_upsample_video(latent, video_encoder, upsampler):
    if _latent_stats("Stage 1 latent", latent):
        # Fail fast: NaN here always ends as a black video, no point spending minutes on Stage 2 + VAE
        raise RuntimeError("Stage 1 produced NaN latents (FP16 overflow in the transformer). "
                           "As a last resort set MODEL_DTYPE = torch.bfloat16 (slower).")
    in_dtype, in_device = latent.dtype, latent.device
    if UPS_ON_GPU1:
        with torch.cuda.device(1):
            out = _orig_pipeline_upsample(latent=latent.to("cuda:1"), video_encoder=video_encoder, upsampler=upsampler)
    else:
        out = _orig_pipeline_upsample(latent=latent, video_encoder=video_encoder, upsampler=upsampler)
    out = out.to(device=in_device, dtype=in_dtype)
    _latent_stats("Upscaled latent", out)
    return out

def _fast_vae_tiling(height, width):
    # Whole frames when they fit, official LTX overlaps (24 frames / 64 px), longest chunks the
    # T4 allows while the transformer is offloaded. ~217 B per pixel-frame measured for this decoder.
    budget = torch.cuda.get_device_properties(0).total_memory - 1.6e9 - 3.0e9
    bytes_per_pixel_frame = 230
    spatial = None
    tile_area = height * width
    if budget / (bytes_per_pixel_frame * tile_area) < 96:
        spatial = SpatialTilingConfig(tile_size_in_pixels=512, tile_overlap_in_pixels=64)
        tile_area = min(height, 512) * min(width, 512)
    frames = max(32, min(160, int(budget / (bytes_per_pixel_frame * tile_area)) // 8 * 8))
    return TilingConfig(spatial_config=spatial,
                        temporal_config=TemporalTilingConfig(tile_size_in_frames=frames, tile_overlap_in_frames=24))

def _probed_decode(latent, video_decoder, tiling_config=None, *args, **kwargs):
    _latent_stats("Final latent -> VAE", latent)
    latent_tensor = latent[0] if isinstance(latent, list) else latent  # decode clears the list; keep it for a retry
    fast_config = _fast_vae_tiling(kwargs.get("expected_height") or 480, kwargs.get("expected_width") or 832)
    print(f"  🎞️ VAE tiling: {'whole frame' if fast_config.spatial_config is None else '512 px tiles'}, "
          f"{fast_config.temporal_config.tile_size_in_frames}-frame chunks (24 overlap)")
    t_dec = time.time()
    torch.cuda.reset_peak_memory_stats(0)
    try:
        out = _orig_pipeline_decode([latent_tensor] if isinstance(latent, list) else latent_tensor,
                                    video_decoder, fast_config, *args, **kwargs)
    except torch.cuda.OutOfMemoryError:
        print("  ⚠️ VAE out of memory with large chunks, retrying with Wan2GP's default tiling...")
        gc.collect()
        torch.cuda.empty_cache()
        out = _orig_pipeline_decode([latent_tensor] if isinstance(latent, list) else latent_tensor,
                                    video_decoder, tiling_config, *args, **kwargs)
    print(f"  🎞️ VAE decode: {time.time() - t_dec:.1f}s (peak VRAM {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB)")
    return out

_distilled_mod.upsample_video = _probed_upsample_video
_distilled_mod.vae_decode_video_to_tensor = _probed_decode

# ==== 5. DUAL T4 ACCELERATION (GPU T4 x2) ====
AUX_ON_GPU1 = False

if NUM_GPUS > 1:
    print("\n🚀 Dual GPU Active (GPU T4 x2) - Offloading Gemma 3 Text Encoder to GPU 1...")
    try:
        # Move Gemma 3 text encoder to GPU 1
        ltx2_model.text_encoder.to("cuda:1")
        _gemma = ltx2_model.text_encoder.model
        _gemma_lm = _gemma.model if hasattr(_gemma, "model") else _gemma
        if getattr(_gemma, "lm_head", None) is not None:
            _gemma.lm_head = torch.nn.Identity()

        # Token-embedding lookup on CPU (few ms) gives GPU 1 maximum free VRAM
        _embed = _gemma_lm.embed_tokens
        _embed.to("cpu")
        _orig_embed_fwd = _embed.forward

        def _cpu_embed_fwd(input_ids, *args, **kwargs):
            return _orig_embed_fwd(input_ids.to("cpu"), *args, **kwargs).to("cuda:1")

        _embed.forward = _cpu_embed_fwd
        _gemma.__class__ = type(_gemma.__class__.__name__, (_gemma.__class__,),
                                {"device": property(lambda self: torch.device("cuda", 1))})
        pipe.pop("text_encoder", None)

        # Wrap encode_raw to send output hidden_states & attention_mask back to cuda:0 for downstream pipeline
        _orig_encode_raw = ltx2_model.text_encoder.encode_raw

        def _dual_gpu_encode_raw(text, padding_side="left"):
            raw_emb = _orig_encode_raw(text, padding_side=padding_side)
            # Device move only: Gemma's hidden states must stay BF16 (values far above FP16's range
            # once squared by the feature extractor's RMS norm; casting them to FP16 gave all-NaN latents)
            hs_cuda0 = tuple(h.to(device="cuda:0") for h in raw_emb.hidden_states)
            _absmax = max(float(h.abs().max()) for h in hs_cuda0)
            print(f"  🔬 Gemma hidden states: {len(hs_cuda0)} layers, dtype={hs_cuda0[0].dtype}, absmax={_absmax:.0f}")
            mask_cuda0 = raw_emb.attention_mask.to("cuda:0")
            return raw_emb.__class__(hs_cuda0, mask_cuda0, raw_emb.padding_side)

        ltx2_model.text_encoder.encode_raw = _dual_gpu_encode_raw

        # Fail-safe device and dtype alignment in postprocess_text_embeddings
        try:
            import models.ltx2.ltx_core.text_encoders.gemma.encoders.base_encoder as _base_enc_mod
            _orig_postprocess = _base_enc_mod.postprocess_text_embeddings

            def _safe_postprocess(embeddings, feat_ext, emb_conn, aud_conn, return_attention_masks=False):
                target_dev = next(feat_ext.parameters()).device if any(True for _ in feat_ext.parameters()) else torch.device("cuda:0")
                safe_embs = []
                for item in embeddings:
                    hs = tuple(h.to(device=target_dev) for h in item.hidden_states)  # keep BF16 (see above)
                    mask = item.attention_mask.to(target_dev)
                    safe_embs.append(item.__class__(hs, mask, item.padding_side))
                return _orig_postprocess(safe_embs, feat_ext, emb_conn, aud_conn, return_attention_masks=return_attention_masks)

            _base_enc_mod.postprocess_text_embeddings = _safe_postprocess
            print("  [GPU T4 x2] ✅ postprocess_text_embeddings device & dtype alignment active.")
        except Exception as _pe:
            print(f"  [GPU T4 x2] ⚠️ Postprocess patch note: {_pe}")

        AUX_ON_GPU1 = True
        _free1 = torch.cuda.mem_get_info(1)[0] / 1024**3
        print(f"  [GPU T4 x2] ✅ Gemma 3 text encoder resident on GPU 1 ({_free1:.1f} GB free on GPU 1).")
        print("  [GPU T4 x2] ✅ GPU 0 is dedicated to the Transformer and VAE decoder.")
    except Exception as e:
        print(f"  [GPU T4 x2] ⚠️ Dual GPU setup notice: {e}, falling back to single GPU mode")
        AUX_ON_GPU1 = False

# The upsampler (between the stages) and the video encoder (reference image, once per stage) would
# each evict the transformer from GPU 0 under mmgp. Both fit next to Gemma on GPU 1 in FP32.
if AUX_ON_GPU1 and pipe.get("spatial_upsampler") is not None:
    with torch.cuda.device(1):
        torch.cuda.empty_cache()
    try:
        for _key in ("spatial_upsampler", "video_encoder"):
            pipe[_key].to("cuda:1")
        for _key in ("spatial_upsampler", "video_encoder"):
            pipe.pop(_key)
        UPS_ON_GPU1 = True
        print(f"  [GPU T4 x2] ✅ Spatial upsampler + video encoder resident on GPU 1 ({torch.cuda.mem_get_info(1)[0] / 1024**3:.1f} GB free).")
    except Exception as e:
        print(f"  [GPU T4 x2] ⚠️ Upsampler / encoder stay on GPU 0 ({e})")
        for _key in ("spatial_upsampler", "video_encoder"):
            getattr(ltx2_model, _key).to("cpu")
        with torch.cuda.device(1):
            torch.cuda.empty_cache()

sys.stdout.flush()

# ==== 6. APPLY MMGP OFFLOADING ON GPU 0 ====
# When Gemma 3 is on GPU 1, transformer budget on GPU 0 can be increased to 11000 MB!
# This keeps the ~11.8 GB Q4_K_M model resident in VRAM during diffusion (no PCIe streaming).
transformer_budget = 11000 if AUX_ON_GPU1 else 6000
print(f"\nApplying mmgp Profile 4 (Transformer VRAM budget: {transformer_budget} MB)...")
sys.stdout.flush()

budgets = {
    "transformer":       transformer_budget,
    "vae":               3500,   # FP32 video decoder (1.5 GB) + working memory
    "spatial_upsampler": 2000,
    "video_encoder":     2000,
    "audio_encoder":     1000,
    "audio_decoder":     1000,
    "vocoder":           500,
    "*":                 1000,
}
if not AUX_ON_GPU1:
    budgets["text_encoder"] = 1500
if UPS_ON_GPU1:
    budgets.pop("spatial_upsampler")
    budgets.pop("video_encoder")

offload.profile(
    pipe,
    profile_no=4,
    quantizeTransformer=False,
    convertWeightsFloatTo=torch.float16,
    budgets=budgets,
)
offload.shared_state["_attention"] = "sdpa"

print("\n✅ Setup complete! Distilled 1.1 Text/Image-to-Video pipeline active.")
sys.stdout.flush()

# ==== 7. HELPER FUNCTIONS ====
AUTO_ASPECT = "Auto (match input image)"

def get_resolution(base_res_str, aspect_ratio_str, ref_image=None):
    base_resolutions = {"1080p": 1088, "720p": 704, "540p": 544, "480p": 480}
    ratios = {
        "16:9 Landscape": 16/9, "4:3 Standard": 4/3,
        "1:1 Square": 1.0, "3:4 Portrait": 3/4, "9:16 Portrait": 9/16,
    }
    base = base_resolutions.get(base_res_str, 704)
    if aspect_ratio_str == AUTO_ASPECT and ref_image is not None:
        ratio = ref_image.width / ref_image.height  # no center-crop of the portrait / start frame
    else:
        ratio = ratios.get(aspect_ratio_str, 16/9)
    if ratio >= 1.0:
        height = base
        width = int(base * ratio)
    else:
        width = base
        height = int(base / ratio)
    return (width // 32) * 32, (height // 32) * 32

# ==== 8. VIDEO GENERATION INFERENCE FUNCTION ====
@torch.inference_mode()
def Video_Generation(prompt, input_image_start, input_image_end, seed, duration_dropdown,
                     resolution_dropdown, aspect_ratio_dropdown, progress=gr.Progress()):
    try:
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        progress(0, desc="Starting...")

        duration_map = {
            "2 Seconds (49 frames)":  49,
            "3 Seconds (73 frames)":  73,
            "5 Seconds (121 frames)": 121,
            "8 Seconds (193 frames)": 193,
            "10 Seconds (241 frames)": 241,
            "15 Seconds (361 frames)": 361,
        }
        frame_rate = 24.0
        num_frames = duration_map.get(duration_dropdown, 121)

        if seed is None or seed < 0:
            seed = random.randint(0, 2**32 - 1)
        seed = int(seed)

        image_start = Image.open(input_image_start).convert("RGB") if input_image_start is not None else None
        image_end = Image.open(input_image_end).convert("RGB") if input_image_end is not None else None
        width, height = get_resolution(resolution_dropdown, aspect_ratio_dropdown,
                                       ref_image=image_start if image_start is not None else image_end)

        if image_start is not None and image_end is not None:
            mode = "First + Last Frame"
        elif image_start is not None:
            mode = "Image-to-Video"
        elif image_end is not None:
            mode = "Last Frame"
        else:
            mode = "Text-to-Video"

        free_vram0 = torch.cuda.mem_get_info(0)[0] / 1024**3
        free_vram1 = (torch.cuda.mem_get_info(1)[0] / 1024**3) if NUM_GPUS > 1 else 0
        ram_avail = psutil.virtual_memory().available / 1024**3

        print(f"\n{'='*65}")
        print(f"🎬 Generating [{mode}]: {width}x{height}, {num_frames} frames, Seed={seed}")
        print(f"  VRAM Free: GPU0={free_vram0:.2f} GB" + (f", GPU1={free_vram1:.2f} GB" if NUM_GPUS > 1 else ""))
        print(f"  RAM Free: {ram_avail:.1f} GB")
        print(f"  Prompt: {prompt[:100]}{'...' if len(prompt) > 100 else ''}")
        print(f"{'='*65}")
        sys.stdout.flush()

        total_steps = [8]
        current_step = [0]
        current_pass = [1]

        def cb(step, latent, is_start, override_num_inference_steps=None, pass_no=None, **kwargs):
            if kwargs.get("progress_unit") is not None:
                return  # Gemma layer / VAE tile progress, not a diffusion step
            if is_start:
                if override_num_inference_steps is not None:
                    total_steps[0] = override_num_inference_steps
                if pass_no is not None:
                    current_pass[0] = pass_no
                current_step[0] = 0
                return
            current_step[0] += 1
            stage_name = "Stage 1 (half-res)" if current_pass[0] == 1 else ("Stage 2 (refine)" if current_pass[0] == 2 else "Diffusion")
            free_v = torch.cuda.mem_get_info(0)[0] / 1024**3
            print(f"  [{stage_name}] step {current_step[0]}/{total_steps[0]} | GPU0 VRAM free: {free_v:.2f} GB")
            sys.stdout.flush()
            frac = current_step[0] / max(total_steps[0], 1)
            frac = 0.73 + 0.22 * frac if current_pass[0] == 2 else frac * 0.73
            progress(min(frac, 0.95), desc=f"{stage_name}: {current_step[0]}/{total_steps[0]}")

        _stage_labels = {
            "VAE Encoding":  ("🎞️ VAE Encoding input frames...",    0.05),
            "VAE Decoding":  ("🎬 VAE Decoding latents to video...", 0.88),
            "Upsampling":    ("🔭 Spatial upsampling latents...",    0.80),
        }
        _t = [time.time()]

        def set_progress_status(status: str):
            dt = time.time() - _t[0]
            _t[0] = time.time()
            label, frac = _stage_labels.get(status, (f"⏳ {status}...", 0.85))
            print(f"  [{status}] {label} (+{dt:.1f}s)")
            sys.stdout.flush()
            progress(frac, desc=label)

        gen_kwargs = dict(
            input_prompt=prompt,
            image_start=image_start,
            height=height,
            width=width,
            frame_num=num_frames,
            fps=frame_rate,
            seed=seed,
            callback=cb,
            set_progress_status=set_progress_status,
            VAE_tile_size=512,          # replaced by the larger whole-frame plan in _probed_decode
            input_video_strength=1.0,   # start / end frames are hard constraints
            denoising_strength=1.0,
            guide_scale=1.0,            # distilled model: guidance is baked in (Wan2GP uses 1.0)
            audio_cfg_scale=1.0,
            alt_guide_scale=1.0,
            guide_phases=2,             # half-res pass, 2x latent upscale, refine pass
            n_prompt="",
            video_prompt_type="",
            audio_prompt_type="",
        )
        if image_end is not None:
            gen_kwargs["image_end"] = image_end

        t_start = time.time()
        result = ltx2_model.generate(**gen_kwargs)
        total_gen_time = time.time() - t_start
        print(f"  [Pipeline Completed in {total_gen_time:.1f}s]")
        progress(0.96, desc="✅ Finalizing video output...")
        sys.stdout.flush()

        if result is None:
            return None, "Generation failed or was interrupted."

        audio_data = None
        audio_sr = None
        if isinstance(result, dict):
            video_tensor = result.get("x")
            audio_data = result.get("audio")
            audio_sr = result.get("audio_sampling_rate", 24000)
        elif isinstance(result, tuple):
            video_tensor = result[0]
            if len(result) > 1: audio_data = result[1]
            if len(result) > 2: audio_sr = result[2]
        else:
            video_tensor = result

        if video_tensor is None or not torch.is_tensor(video_tensor):
            return None, f"❌ No video tensor produced. Got: {type(video_tensor)}"

        video_tensor = video_tensor.cpu()
        gc.collect()
        torch.cuda.empty_cache()

        out_path = os.path.join(OUTPUT_DIR, f"ltx23_11_{time.strftime('%Y%m%d_%H%M%S')}_seed{seed}.mp4")
        if video_tensor.dtype == torch.uint8:
            save_video(tensor=video_tensor.unsqueeze(0), save_file=out_path, fps=frame_rate, nrow=1)
        else:
            save_video(tensor=video_tensor.unsqueeze(0).float() / 127.5 - 1.0, save_file=out_path, fps=frame_rate,
                       nrow=1, normalize=True, value_range=(-1, 1))

        # LTX-2 generates a matching soundtrack with every video: mux it in
        if audio_data is not None:
            try:
                audio_tmp = tempfile.mktemp(suffix=".wav")
                if isinstance(audio_data, np.ndarray):
                    audio_np = audio_data
                    if audio_np.ndim == 2 and audio_np.shape[0] <= 2:
                        audio_np = audio_np.T
                    sf.write(audio_tmp, audio_np, int(audio_sr or 24000))
                elif torch.is_tensor(audio_data):
                    import torchaudio
                    cpu_audio = audio_data.cpu().float()
                    if cpu_audio.dim() == 1: cpu_audio = cpu_audio.unsqueeze(0)
                    if cpu_audio.dim() == 3: cpu_audio = cpu_audio.squeeze(0)
                    torchaudio.save(audio_tmp, cpu_audio, int(audio_sr or 24000))

                final_path = out_path.replace(".mp4", "_with_audio.mp4")
                subprocess.run([
                    "ffmpeg", "-y", "-i", out_path, "-i", audio_tmp,
                    "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
                    "-shortest", final_path
                ], check=True, capture_output=True)
                if os.path.exists(final_path) and os.path.getsize(final_path) > 0:
                    os.remove(out_path)  # keep only the version with audio in the outputs folder
                    out_path = final_path
                    print(f"  ✅ Audio muxed: {out_path}")
                if os.path.exists(audio_tmp):
                    os.remove(audio_tmp)
            except Exception as e:
                print(f"  ⚠️ Audio mux note: {e}")

        del video_tensor
        gc.collect()
        torch.cuda.empty_cache()
        progress(1.0, desc="Done!")
        return out_path, f"✅ Done in {total_gen_time:.1f}s! {mode} | Seed: {seed} | {width}x{height} | {num_frames} frames"

    except Exception as e:
        traceback.print_exc()
        gc.collect()
        torch.cuda.empty_cache()
        return None, f"❌ Error: {str(e)}"

# ==== 9. AIQUEST BRANDED GRADIO INTERFACE ====
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 12px 0; }
.brand-footer { text-align: center; color: #6b7280; font-size: 12px; margin-top: 24px; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎬 LTX-2.3 22B Distilled 1.1 Video Generator</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Kaggle GPU T4 x2 Edition</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""

_gpu_layout = "Gemma + upsampler on GPU 1" if UPS_ON_GPU1 else ("Gemma on GPU 1" if AUX_ON_GPU1 else "single GPU")
STATUS_HTML = f'<div class="status-line">⚡ Distilled 1.1 GGUF Q4_K_M (FP16) &nbsp;|&nbsp; {_gpu_layout} &nbsp;|&nbsp; FP32 VAE</div>'

FOOTER_HTML = """
<div class="brand-footer">⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved</div>
"""

with gr.Blocks(theme=gr.themes.Soft(), css=CSS, title="LTX-2.3 Distilled 1.1 Video Generator | AIQUEST Academy") as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)

    gr.Markdown(
        "💡 **Tips:** write detailed prompts (subject, action, setting, camera, lighting, sound). "
        "Leave the images empty for Text-to-Video; add a start and/or end frame for Image-to-Video."
    )

    with gr.Column():
        prompt = gr.Textbox(
            label="🎨 Prompt", lines=3,
            placeholder="A majestic eagle soaring over snowy mountain peaks at golden hour, wind rushing past, cinematic tracking shot..."
        )

        with gr.Accordion("🖼️ Image to Video (Optional)", open=False):
            with gr.Row():
                input_image_start = gr.Image(type="filepath", label="🎬 Start Frame (First Frame)")
                input_image_end = gr.Image(type="filepath", label="🎬 End Frame (Last Frame, Optional)")
            gr.Markdown(
                "*Start frame only = Image-to-Video. Start + end = First/Last frame interpolation. "
                "End only = the video lands on that frame. Neither = Text-to-Video.*"
            )

        with gr.Row():
            seed = gr.Number(label="🎲 Seed (-1 for Random)", value=-1, precision=0)
            duration_dropdown = gr.Dropdown(
                label="⏱️ Duration",
                choices=[
                    "2 Seconds (49 frames)",
                    "3 Seconds (73 frames)",
                    "5 Seconds (121 frames)",
                    "8 Seconds (193 frames)",
                    "10 Seconds (241 frames)",
                    "15 Seconds (361 frames)",
                ],
                value="5 Seconds (121 frames)",
            )

        with gr.Row():
            resolution_dropdown = gr.Dropdown(
                label="📐 Base Resolution",
                choices=["720p", "540p", "480p"],
                value="720p",
            )
            aspect_ratio_dropdown = gr.Dropdown(
                label="📏 Aspect Ratio",
                choices=[AUTO_ASPECT, "16:9 Landscape", "4:3 Standard", "1:1 Square", "3:4 Portrait", "9:16 Portrait"],
                value=AUTO_ASPECT,
                info="Auto follows the start/end image (16:9 for Text-to-Video)",
            )

        with gr.Row():
            gen_btn  = gr.Button("🎬 Generate Video", variant="primary",   size="lg", elem_id="gen-btn")
            stop_btn = gr.Button("🛑 Stop",           variant="secondary", size="lg", elem_id="stop-btn")
            clear_btn= gr.Button("🗑️ Clear",          variant="secondary", size="lg", elem_id="clear-btn")

        video_out = gr.Video(label="🎥 Generated Video")
        latest_btn = gr.Button("📂 Load Latest Video (use if the UI showed an error)", variant="secondary")
        status_out = gr.Textbox(label="ℹ️ Status", interactive=False)

        def load_latest_video():
            videos = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.mp4")), key=os.path.getmtime)
            if not videos:
                return None, f"No videos in {OUTPUT_DIR} yet."
            return videos[-1], f"📁 Loaded latest video: {videos[-1]}"

        latest_btn.click(fn=load_latest_video, outputs=[video_out, status_out])

        gen_event = gen_btn.click(
            fn=Video_Generation,
            inputs=[prompt, input_image_start, input_image_end, seed, duration_dropdown,
                    resolution_dropdown, aspect_ratio_dropdown],
            outputs=[video_out, status_out],
        )
        stop_btn.click(fn=None, cancels=[gen_event])
        clear_btn.click(
            fn=lambda: (None, None, None, None, "", -1),
            outputs=[input_image_start, input_image_end, video_out, status_out, prompt, seed],
        )

    gr.HTML(FOOTER_HTML)

# ==== 10. CLOUDFLARE QUICK TUNNEL (second public link in case the Gradio share link fails) ====
SERVER_PORT = 7860

def start_cloudflare_tunnel(port):
    import re
    import stat
    import threading
    import urllib.request
    binary = "/tmp/cloudflared"
    try:
        if not os.path.exists(binary):
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
            os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
        tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel unavailable: {e}")
        return

    def _watch():
        for line in tunnel.stdout:
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if match:
                print(f"* Cloudflare tunnel URL: {match.group(0)}  (use this if the Gradio link does not load)")
                sys.stdout.flush()
                break
        for _ in tunnel.stdout:  # keep draining so cloudflared never blocks on a full pipe
            pass

    threading.Thread(target=_watch, daemon=True).start()

print("\nLaunching Gradio Interface...")
sys.stdout.flush()
start_cloudflare_tunnel(SERVER_PORT)
demo.queue()
demo.launch(server_name="127.0.0.1", server_port=SERVER_PORT, share=True, inline=False, debug=True,
            show_error=True, max_threads=1, ssr_mode=False, allowed_paths=[OUTPUT_DIR])



Writing run_ltx_t2v.py


---
## Step 5: Launch!

Runs the generation script. Watch for the **Gradio** and **Cloudflare** public URLs in the output.

In [ ]:
!cd /kaggle/working && python -u run_ltx_t2v.py 2>&1

  [GPU] sm_75 detected - native CUDA mode, no patches needed
GPU: Tesla T4
Compute Capability: (7, 5)
VRAM: 14.6 GB
RAM: 31.3 GB total, 27.3 GB available

Loading LTX-2.3 22B Distilled 1.1 (GGUF Q4_K_M)...
2026-09-25 01:29:40.524491: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790299780.762327     732 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790299780.847381     732 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790299781.364692     732 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790299781.364765     732 computation_placer

---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---